In [0]:
%run ./00_configuration

## Document contract and utilities (domain-agnostic)

Pure Python, no Spark writes in this section. `parse_document` validates any source file against the metadata contract configured in `00_configuration` (`CORE_METADATA_FIELDS` + `DOMAIN_ATTRIBUTES`), so retargeting the framework to another domain requires a configuration change only — the contract code stays the same. Utilities cover hashing, section-aware chunking (parameters from configuration), registry validation, the keyword fallback, extractive responses, and embedding parsing/cosine helpers.

In [0]:
"""Pure Python document contract, chunking, validation, and fallback utilities."""
import hashlib


def digest(text):
    """SHA-256 hex digest of a UTF-8 string.

    Defines content identity for versions, chunk IDs, and quarantine
    deduplication: identical text always maps to the same digest.
    """
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


def parse_document(text):
    """Parse and validate one file against the embedded-metadata contract.

    Raises ValueError with the first failing rule (bad header, missing or
    non-string metadata field, non-ISO date, empty body) so ingestion can
    quarantine the file with a readable reason. Returns metadata, body, and
    content_hash of the FULL file text (metadata and body): any edit anywhere
    yields a different hash, so a file can never silently replace a version.
    """
    match = re.match(r'\A---metadata\s*\n(.*?)\n---\s*\n(.*)\Z', text, re.S)
    if not match:
        raise ValueError('Expected ---metadata, JSON block, ---, then document body')
    meta, body = json.loads(match[1]), match[2].strip()
    required = METADATA_FIELDS  # document identity + configured domain attributes
    for key in required:
        if not isinstance(meta.get(key), str) or not meta[key].strip():
            raise ValueError(f'Missing or invalid {key}')
    date.fromisoformat(meta['document_date'])
    if not body:
        raise ValueError('Empty document body')
    return {**{k: meta[k].strip() for k in required}, 'body': body, 'content_hash': digest(text)}


def chunk_document(record, max_words=None, overlap=None):
    """Split one accepted version into section-aware chunks.

    Splits on top-level '#' headings; long sections are windowed to at most
    max_words body words (CHUNK_MAX_WORDS by default) with CHUNK_OVERLAP_WORDS
    overlap; the record's title is prepended to every chunk's searchable text.
    chunk_id digests business key + version + content_hash + section and window
    position, so identical content always yields the same chunk row (idempotent
    MERGEs) while any content change produces new chunk IDs. 'title' belongs to
    the default DOMAIN_ATTRIBUTES; a deployment that removes it must adapt this
    prepend step.
    """
    max_words = CHUNK_MAX_WORDS if max_words is None else max_words
    overlap = CHUNK_OVERLAP_WORDS if overlap is None else overlap
    if not 0 <= overlap < max_words:
        raise ValueError('Invalid overlap')
    chunks = []
    for section_number, section in enumerate(re.split(r'(?m)(?=^# )', record['body'])):
        words = section.split()
        if not words:
            continue
        for start in range(0, len(words), max_words-overlap):
            text = record['title'] + '\n' + ' '.join(words[start:start+max_words])
            chunks.append(dict(
                **{k:v for k,v in record.items() if k != 'body'},
                chunk_id=digest(record['document_id']+'|'+record['version_id']+'|'+
                                record['content_hash']+'|'+str(section_number)+'|'+str(start)),
                section_heading=section.strip().splitlines()[0].lstrip('# '),
                chunk_text=text))
            if start + max_words >= len(words):
                break
    return chunks


def validate_candidates(candidates, registry):
    """Never pass stale text to a model. registry maps business ID to version/hash."""
    return [c for c in candidates
            if registry.get(c['document_id']) == (c['version_id'], c['content_hash'])]


def rank_chunks(question, chunks, limit=5):
    """Standard-library TF-IDF cosine retrieval; no model or vector service."""
    import math
    from collections import Counter
    stop = {'a','an','the','is','are','what','which','for','of','to','in','and','please','tell','me'}
    def terms(s):
        return [w for w in re.findall(r'[a-z0-9]+', s.lower()) if w not in stop]
    query = Counter(terms(question))
    if not query or not chunks:
        return []
    docs = [Counter(terms(c['chunk_text'])) for c in chunks]
    df = Counter(w for doc in docs for w in doc)
    n = len(docs)
    def vector(counts):
        return {w:(1+math.log(count))* (math.log((1+n)/(1+df[w]))+1)
                for w,count in counts.items()}
    q = vector(query)
    qnorm = math.sqrt(sum(v*v for v in q.values()))
    scored = []
    for chunk, counts in zip(chunks,docs):
        v = vector(counts)
        norm = math.sqrt(sum(x*x for x in v.values()))
        score = sum(q.get(w,0)*x for w,x in v.items())/(qnorm*norm) if norm else 0
        if score > 0:
            scored.append(dict(chunk, retrieval_score=score))
    return sorted(scored,key=lambda c:(-c['retrieval_score'],c['chunk_id']))[:limit]


def extractive_response(question, evidence):
    """Return source excerpts, not generated facts or an LLM answer."""
    if not evidence:
        return {'answer':'No matching current document excerpt was found.', 'sources':[], 'mode':'EXTRACTIVE'}
    sources, excerpts = [], []
    for i,e in enumerate(evidence,1):
        source = {key:e[key] for key in ('document_id','version_id','document_date','filename','source_path','section_heading')}
        sources.append(source)
        excerpts.append(f"[{i}] {e['chunk_text']}")
    return {'answer':'Current document excerpts:\n\n'+'\n\n'.join(excerpts),
            'sources':sources,'mode':'EXTRACTIVE'}


def parse_embedding(value):
    """Normalize common SQL embedding results to a finite, nonzero float vector."""
    import math
    if isinstance(value,str): value=json.loads(value)
    if isinstance(value,dict):
        if 'embedding' in value: value=value['embedding']
        elif 'data' in value and value['data']: value=value['data'][0]['embedding']
    if not isinstance(value,(list,tuple)) or not value:
        raise ValueError('Embedding response is not a nonempty vector')
    vector=[float(x) for x in value]
    if not all(math.isfinite(x) for x in vector) or not any(vector):
        raise ValueError('Embedding contains nonfinite values or is zero')
    return vector


def cosine_similarity(a,b):
    import math
    if len(a)!=len(b): raise ValueError('Embedding dimension mismatch')
    denom=math.sqrt(sum(x*x for x in a)*sum(x*x for x in b))
    return sum(x*y for x,y in zip(a,b))/denom if denom else 0.0

## Storage layer: Delta tables, manifest, registry, and curated views

`ensure_storage()` creates nine Delta tables (created only when absent, so reruns are safe and history is preserved) and eight views:

* `document_versions` — immutable accepted version history (metadata, body, hash, source file)
* `document_chunks` — chunk history for **every** accepted version, active or not
* `active_document_chunks` — the retrieval source, refreshed to hold active-version chunks only (Change Data Feed enabled so a Delta Sync index could consume it later)
* `chunk_embeddings` — persisted embeddings keyed by `(chunk_id, model)`
* `quarantined_documents` — rejected source files with a reason and observation timestamp
* `retrieval_audit` — one row per retrieval request (question, mode, candidate and accepted refs, rejected count) — the original audit trail, still written on every search
* `request_log` — one row per retrieval/answer request with the request_id that links both halves: modes, configured models, accepted version refs, rejected stale-candidate count, latencies, errors, and fallback reasons (nullable where not captured)
* `embedding_log` — application-submitted embedding requests and cache-reuse observations (document vs query, submitted text) — not provider-internal inference counts
* `evaluation_results_raw` — append-only evaluation results from the `60_evaluation` harness (run_id, test_id, category, assessment type, expected/actual, status, diagnostics, simulated flag)
* `document_manifest` (view) — version history with rank and computed `is_active`
* `active_document_registry` (view) — current document ID, version, hash, and date
* `document_inventory` (curated view) — one row per accepted version with `version_status` = ACTIVE / FUTURE_DATED / HISTORICAL_INACTIVE and pre-aggregated chunk/embedding counts
* `quarantine_observations` (curated view) — one row per quarantine record, labeled as an **observation** (resolution not tracked, `is_resolved` is NULL)
* `retrieval_activity` (curated view) — one row per request; new traffic fully captured, legacy audit history shown once with NULLs (never double counted); `log_context` separates demo/evaluation/simulated/operational traffic
* `embedding_activity` (curated view) — one row per candidate text per request; distinguishes submitted embeddings from cache reuse
* `evaluation_results` / `evaluation_run_summary` (curated views) — evaluation detail (grain: run_id + test_id) and per-run summary; the pass rate counts executed assessments only

Table schemas are generated from the metadata contract in `00_configuration`, so adding a domain attribute grows every history table consistently. Every curated view documents its grain and joins (view comment) to prevent double counting.

In [0]:
def ensure_storage():
    """Create the project's Delta tables and views when absent (idempotent).

    Tables are created only when absent, so reruns never duplicate or overwrite
    history; the manifest and registry views are replaced so their definition
    always matches the current code.
    """
    for name, schema in [
        ("document_versions", VERSION_SCHEMA),
        ("document_chunks", CHUNK_SCHEMA),
        ("active_document_chunks", CHUNK_SCHEMA),
        ("chunk_embeddings", "chunk_id string, model string, embedding_json string"),
        ("quarantined_documents", QUARANTINE_SCHEMA),
        ("retrieval_audit", AUDIT_SCHEMA),
        ("request_log", REQUEST_LOG_SCHEMA),
        ("embedding_log", EMBEDDING_LOG_SCHEMA),
        ("evaluation_results_raw", EVAL_RESULTS_SCHEMA)]:
        if not spark.catalog.tableExists(table(name)):
            spark.createDataFrame([], schema).write.format("delta").saveAsTable(table(name))
    spark.sql(f"ALTER TABLE {table('active_document_chunks')} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
    # Manifest is a view over immutable, accepted versions: activation is always
    # recomputed from history, never stored as a flag in a file. Equal-date ambiguity
    # is rejected during ingest BEFORE rows reach this view. Ordering: versions with an
    # eligible date first (future versions cannot win), then newest business date, then
    # version_id as a deterministic tiebreak. Arrival time and filename never participate.
    spark.sql(f"""CREATE OR REPLACE VIEW {table('document_manifest')} AS
     SELECT *, version_rank = 1 AND document_date <= CAST(current_date() AS STRING) AS is_active
     FROM (
       SELECT *, ROW_NUMBER() OVER (PARTITION BY document_id
         ORDER BY CASE WHEN document_date <= CAST(current_date() AS STRING) THEN 0 ELSE 1 END,
                  document_date DESC, version_id) AS version_rank
       FROM {table('document_versions')}
     )""")
    # Single source of truth: retrieval must match every candidate against this
    # registry's (document_id, version_id, content_hash) before any evidence reaches a model.
    spark.sql(f"""CREATE OR REPLACE VIEW {table('active_document_registry')} AS
     SELECT document_id, version_id, content_hash, document_date
     FROM {table('document_manifest')} WHERE is_active""")
    # --- Curated, documented views for analytics and future dashboards ---------
    # Each view states its GRAIN in the view comment; every join is pre-aggregated
    # to that grain so no row fanout (and no double counting) is possible.
    #
    # document_inventory — grain: exactly one row per accepted (document_id,
    # version_id) from the manifest. version_status distinguishes future-dated
    # versions (business date in the future, so not yet eligible) from historical
    # inactivity (superseded or late-arriving history kept for traceability).
    spark.sql(f"""CREATE OR REPLACE VIEW {table('document_inventory')} AS
     SELECT m.document_id, m.version_id, m.document_date, m.title, m.crop, m.region, m.topic,
            m.filename, m.source_path, m.ingested_at, m.is_active,
            CASE WHEN m.is_active THEN 'ACTIVE'
                 WHEN m.document_date > CAST(current_date() AS STRING) THEN 'FUTURE_DATED'
                 ELSE 'HISTORICAL_INACTIVE' END AS version_status,
            COALESCE(c.chunk_count, 0) AS chunk_count,
            COALESCE(e.embedded_chunk_count, 0) AS embedded_chunk_count
     FROM {table('document_manifest')} m
     LEFT JOIN (SELECT document_id, version_id, COUNT(*) AS chunk_count
                FROM {table('document_chunks')} GROUP BY document_id, version_id) c
       ON m.document_id = c.document_id AND m.version_id = c.version_id
     LEFT JOIN (SELECT dc.document_id, dc.version_id,
                       COUNT(DISTINCT ce.chunk_id) AS embedded_chunk_count
                FROM {table('document_chunks')} dc
                JOIN {table('chunk_embeddings')} ce ON dc.chunk_id = ce.chunk_id
                GROUP BY dc.document_id, dc.version_id) e
       ON m.document_id = e.document_id AND m.version_id = e.version_id""")
    spark.sql(f"COMMENT ON VIEW {table('document_inventory')} IS 'GRAIN: one row per accepted (document_id, version_id). version_status: ACTIVE (current), FUTURE_DATED (business date in the future, not yet eligible — not the same as retired), HISTORICAL_INACTIVE (superseded or late-arriving history kept for traceability). chunk_count/embedded_chunk_count are pre-aggregated to this grain — no fanout. Created by 10_document_framework.'")
    # quarantine_observations — grain: one row per quarantine record. Records are
    # OBSERVATIONS captured at ingestion time, not open issues: the source may
    # since have been removed or corrected, and resolution is NOT tracked here
    # (is_resolved is deliberately NULL until that is implemented).
    spark.sql(f"""CREATE OR REPLACE VIEW {table('quarantine_observations')} AS
     SELECT source_path, content_hash, reason, observed_at,
            CAST(NULL AS BOOLEAN) AS is_resolved, 'observation' AS record_type
     FROM {table('quarantined_documents')}""")
    spark.sql(f"COMMENT ON VIEW {table('quarantine_observations')} IS 'GRAIN: one row per ingestion quarantine record, deduplicated on (source_path, content_hash, reason). Records are historical OBSERVATIONS at the moment of ingest, not unresolved issues; the file may since have been removed or fixed. Resolution status is not tracked in this project (is_resolved is always NULL). Created by 10_document_framework.'")
    # retrieval_activity — grain: exactly ONE row per retrieval/answer request.
    # New traffic (request_log, written by 30_retrieval_answers since the Stage 2
    # logging) carries the full capture set. Historical audit rows whose
    # request_id has no request_log row appear exactly once, with NULLs for
    # measurements that were never captured. Audit rows that DO match a
    # request_log row are excluded — that is what prevents double counting.
    # log_context separates demo / evaluation / operational traffic.
    spark.sql(f"""CREATE OR REPLACE VIEW {table('retrieval_activity')} AS
     SELECT request_id, question, log_context, retrieval_mode, answer_mode,
            embedding_model, chat_model, accepted_versions_json, rejected_stale_candidates,
            retrieval_latency_ms, answer_latency_ms, total_latency_ms,
            error, fallback_reason, started_at, completed_at, 'request_log' AS record_source
     FROM {table('request_log')}
     UNION ALL
     SELECT a.request_id, a.question, CAST(NULL AS STRING) AS log_context,
            a.mode AS retrieval_mode, CAST(NULL AS STRING) AS answer_mode,
            CAST(NULL AS STRING) AS embedding_model, CAST(NULL AS STRING) AS chat_model,
            CAST(NULL AS STRING) AS accepted_versions_json, a.rejected_count AS rejected_stale_candidates,
            CAST(NULL AS LONG) AS retrieval_latency_ms, CAST(NULL AS LONG) AS answer_latency_ms,
            CAST(NULL AS LONG) AS total_latency_ms, CAST(NULL AS STRING) AS error,
            CAST(NULL AS STRING) AS fallback_reason, CAST(NULL AS STRING) AS started_at,
            a.observed_at AS completed_at, 'legacy_audit' AS record_source
     FROM {table('retrieval_audit')} a
     WHERE NOT EXISTS (SELECT 1 FROM {table('request_log')} r WHERE r.request_id = a.request_id)""")
    spark.sql(f"COMMENT ON VIEW {table('retrieval_activity')} IS 'GRAIN: one row per retrieval/answer request (never more). record_source=request_log rows carry the full capture: modes, configured models, accepted version refs, rejected stale candidates, latencies, error, fallback_reason, and log_context (demo/evaluation/operational). record_source=legacy_audit rows are pre-logging history from retrieval_audit with NULLs for fields never captured; audit rows matching a request_log row are excluded so no request is counted twice. Created by 10_document_framework.'")
    # embedding_activity — grain: one row per candidate text considered for
    # embedding within one request (many-to-one to retrieval_activity on
    # request_id). text_submitted is NULL on cache-reused rows (nothing was
    # sent). These are application-submitted requests and reuse observations —
    # they do NOT equal provider-internal inference counts.
    spark.sql(f"""CREATE OR REPLACE VIEW {table('embedding_activity')} AS
     SELECT request_id, log_context, embedding_kind, text_submitted, text_chars,
            cache_reused, embedding_model, submitted_at
     FROM {table('embedding_log')}""")
    spark.sql(f"COMMENT ON VIEW {table('embedding_activity')} IS 'GRAIN: one row per candidate text per request; many-to-one join to retrieval_activity on request_id. embedding_kind: document or query. text_submitted is NULL when cache_reused=true (vector came from chunk_embeddings or the session query cache — nothing was sent). Counts application-submitted embedding requests and reuse observations, NOT provider-internal inference counts. Created by 10_document_framework.'")
    # Evaluation storage backs the 60_evaluation harness: evaluation_results_raw
    # is the append-only detail table; the two curated views expose detail and
    # per-run summary. pass_rate_executed counts PASS/FAIL only — NOT_EXECUTED
    # (including pending manual review) can never count as a pass.
    spark.sql(f"""CREATE OR REPLACE VIEW {table('evaluation_results')} AS
     SELECT run_id, test_id, category, assessment_type, expected_outcome, actual_outcome,
            status, simulated, diagnostics, evaluated_at,
            status != 'NOT_EXECUTED' AS is_executed,
            assessment_type != 'manual_review' AND status != 'NOT_EXECUTED' AS counted_in_pass_rate
     FROM {table('evaluation_results_raw')}""")
    spark.sql(f"COMMENT ON VIEW {table('evaluation_results')} IS 'GRAIN: one row per (run_id, test_id). is_executed and counted_in_pass_rate exclude NOT_EXECUTED and manual_review rows so unsupported or unexecuted assessments can never count as passes. simulated=true rows come from controlled-injection or mocked runs. Created by 10_document_framework.'")
    spark.sql(f"""CREATE OR REPLACE VIEW {table('evaluation_run_summary')} AS
     SELECT run_id, COUNT(*) AS total_tests,
            SUM(CASE WHEN status='PASS' THEN 1 ELSE 0 END) AS passed,
            SUM(CASE WHEN status='FAIL' THEN 1 ELSE 0 END) AS failed,
            SUM(CASE WHEN status='NOT_EXECUTED' THEN 1 ELSE 0 END) AS not_executed,
            SUM(CASE WHEN simulated THEN 1 ELSE 0 END) AS simulated_tests,
            SUM(CASE WHEN assessment_type='deterministic' THEN 1 ELSE 0 END) AS deterministic_tests,
            SUM(CASE WHEN assessment_type='model_based' THEN 1 ELSE 0 END) AS model_based_tests,
            SUM(CASE WHEN assessment_type='manual_review' THEN 1 ELSE 0 END) AS manual_review_tests,
            CASE WHEN SUM(CASE WHEN status IN ('PASS','FAIL') THEN 1 ELSE 0 END)=0 THEN NULL
                 ELSE ROUND(SUM(CASE WHEN status='PASS' THEN 1 ELSE 0 END) * 1.0
                      / SUM(CASE WHEN status IN ('PASS','FAIL') THEN 1 ELSE 0 END), 3) END AS pass_rate_executed,
            MIN(evaluated_at) AS run_started, MAX(evaluated_at) AS run_completed
     FROM {table('evaluation_results_raw')} GROUP BY run_id""")
    spark.sql(f"COMMENT ON VIEW {table('evaluation_run_summary')} IS 'GRAIN: one row per evaluation run (run_id). pass_rate_executed considers only executed assessments (PASS/FAIL); NOT_EXECUTED — including pending manual review — never counts as a pass or a failure. Created by 10_document_framework.'")

ensure_storage()
print("Storage ensured:", f"{CATALOG}.{SCHEMA}")